# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

In [2]:
# Añadido: semilla para reproducibilidad y animaciones en JPEG (notebook más liviano)
random.seed(42)
np.random.seed(42)
plt.rcParams["animation.frame_format"] = "jpeg"

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- **Gymnasium es el ambiente**: define estados, acciones, `reset()`, `step()` y recompensas; equivale a la clase `CheeseWorld`.
- **El agente ya no es una clase**: está repartido entre la Q-table (`Q`, un arreglo NumPy) y las funciones `epsilon_greedy_policy`, `greedy_policy` y `train_q_learning`, que eligen acciones y aplican la actualización de Q-Learning.
- La interfaz es la misma: $(s,a)\rightarrow(s',r,\text{done})$, ahora con `terminated`/`truncated`.


## 2. Crear FrozenLake


In [3]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

- **16 filas**: una por estado (`env.observation_space.n = 16`, rejilla 4×4).
- **4 columnas**: una por acción (Left, Down, Right, Up).
- $Q[s,a]$ es la estimación del retorno descontado esperado al ejecutar la acción $a$ en el estado $s$ y luego seguir la mejor política.


In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- Que todo sea 0 significa que el agente **todavía no sabe nada**, no que las acciones sean malas. En FrozenLake la única recompensa es $+1$ al llegar a la meta, así que 0 es solo el valor inicial.
- Si varias acciones empatan en el máximo, `greedy_policy` **desempata al azar** (`np.random.choice` sobre `np.flatnonzero(Q[s] == max)`). Al inicio todas empatan, de modo que el comportamiento "greedy" es equivalente a uno aleatorio.


## 3. Ejecutar una transición


In [5]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 1
reward     = 0
next_state = 4
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(0,\ 1,\ 0,\ 4)
$$

Es decir: desde $s=0$ se tomó $a=1$ (Down), se recibió $r=0$ y se llegó a $s'=4$ (salida de la celda anterior).

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


**Respuesta:** en Value Iteration no había experiencia: conocíamos el **modelo** $T(s,a,s')$ y $R(s)$ y calculábamos la esperanza sobre **todos** los $s'$. Lo que no teníamos era un **$s'$ observado** (y su $r$) obtenido al actuar: aquí no se conoce $T$ y el agente solo ve la muestra que entrega el ambiente.

*Nota:* la pregunta pide un único elemento; la respuesta más directa es $s'$, porque $r$ también lo daba el modelo como $R(s)$. Se deja indicado porque la redacción admite ambas lecturas.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [6]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


In [7]:
# Añadido: valores de Q en los tres momentos para el estado inicial s = 0
np.set_printoptions(precision=3, suppress=True)
for name, Qx in [("Q_initial", Q_initial), ("Q_early", Q_early), ("Q_trained", Q_trained)]:
    print(f"{name:9s} Q(0,·) = {Qx[0]}  | celdas distintas de 0: {np.count_nonzero(Qx)}")

Q_initial Q(0,·) = [0. 0. 0. 0.]  | celdas distintas de 0: 0
Q_early   Q(0,·) = [0.    0.    0.091 0.   ]  | celdas distintas de 0: 11
Q_trained Q(0,·) = [0.735 0.774 0.774 0.735]  | celdas distintas de 0: 35


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [8]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [9]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 1.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [10]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


**Respuesta:** cambió **solo la Q-table**. Con `Q_initial` todos los valores son 0, así que la política es un random walk: en la ejecución mostrada el agente no llega a la meta (recompensa 0). Tras 50 episodios (`Q_early`) hay 11 valores distintos de 0 y en el estado inicial solo `Right` tiene valor positivo ($Q(0,\text{Right})=0.091$); aun así, en esta ejecución llegó a la meta (recompensa 1). Con `Q_trained` (10 000 episodios) hay 35 valores distintos de 0 y $Q(0,\cdot)=[0.735, 0.774, 0.774, 0.735]$: las acciones que acercan a la meta tienen mayor valor y la política greedy las sigue. El ambiente no cambió; cambiaron los valores $Q(s,a)$ aprendidos.

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** $s=0$ (inicio), con `Q_trained`.

**Valores Q:**

- Left: 0.735
- Down: 0.774
- Right: 0.774
- Up: 0.735

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Interpretación:**

> $\arg\max_a Q(0,a)$ es un **empate entre Down y Right** (0.774); `greedy_policy` elige una de las dos al azar. Ambas inician un camino mínimo de 6 pasos hasta la meta: $0.95^{5}\approx0.774$ es exactamente el valor descontado de una recompensa $+1$ recibida al sexto paso. Left y Up (0.735 ≈ $0.95^6$) no acercan a la meta (el agente choca con el borde y se queda en el lugar), así que cuestan un paso extra.


## 5. Evaluar la política aprendida


In [11]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> Evaluamos para medir **lo que aprendió** la política. La exploración sirve para descubrir mejores acciones durante el entrenamiento; en evaluación solo añadiría acciones aleatorias que no reflejan lo aprendido y bajarían la recompensa. Con la política greedy, el agente logra una recompensa media de **1.000 ± 0.000** en 100 episodios: siempre llega a la meta.


## 6. Experimento: FrozenLake estocástico


In [12]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.362
Std reward : 0.481


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con `is_slippery=True` las transiciones son **estocásticas**: la acción elegida solo se ejecuta con probabilidad 1/3 y con 1/3 el agente se desliza a cada lado perpendicular. La misma $(s,a)$ puede llevar a distintos $s'$, incluso a un agujero. El mejor desempeño posible baja: la política aprendida logra **0.362 ± 0.481** en 500 episodios, frente a 1.000 en el caso determinista.
- **Algoritmo:** la **ecuación de Q-Learning no cambia**: $Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$. Lo que cambia es que cada actualización usa una muestra ruidosa de $s'$; Q-Learning promedia implícitamente esas muestras a lo largo de muchas visitas. Por eso el notebook usa más episodios (20 000) y un decaimiento de $\epsilon$ más lento (0.0005).


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> La estimación del **retorno descontado esperado** de ejecutar la acción $a$ en el estado $s$ y luego actuar de forma óptima (greedy respecto a $Q$).

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la **propia Q-table**, evaluada en el estado siguiente $s'$ **observado**: es el mejor valor que el agente cree poder obtener desde allí (*bootstrapping*). Si $s'$ es terminal, ese término vale 0.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar **exploración y explotación**. Con $Q=0$ al inicio, una política solo greedy repetiría las mismas acciones y podría no encontrar nunca la recompensa; $\epsilon$ obliga a probar acciones nuevas, y al decaer permite explotar lo aprendido.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer $T(s,a,s')$ ni $R$: aprende directamente de las transiciones $(s,a,r,s')$ que observa al interactuar con el ambiente, a diferencia de Value Iteration, que requiere el modelo completo.
